# 01 - ClinVar Veri Keşfi ve Ön İnceleme

Bu çalışma, **Genetik Varyantların Benign veya Pathogenic Olarak Sınıflandırılması** bitirme projesinin ilk adımı olan **ClinVar veri keşfi** aşamasını kapsamaktadır.

---

### Çalışmanın Amacı ve Kapsamı
- **Klinik Etiket Kaynağı:** ClinVar (`variant_summary.txt.gz`) dosyasının yapısını ve içerdiği klinik sınıflandırmaları incelemek.
- **Kanıta Dayalı Keşif:** Herhangi bir veri temizleme (data cleaning), filtreleme (dropping), eksik veri doldurma (imputation) veya makine öğrenmesi modellemesine geçmeden önce verinin ham özelliklerini anlamak.
- **Bilinçli Karar Alma:** Referans montaj seçimi (GRCh37 vs GRCh38), hedef etiket tanımları ve dış kaynak uyumluluğu için somut sayısal kanıt toplamak.

---

### Metodolojik İlkeler
1. **Bellek (RAM) Yönetimi:** Dosya ~9.2 milyon satır ve 43 sütun içerdiğinden, tüm veri RAM'e yüklenmeyecektir. İhtiyaca göre `nrows`, `usecols` ve `chunksize` yöntemleri kullanılacaktır.
2. **Peşin Karar Vermeme:** "GRCh38 daha yeni, doğrudan bunu seçelim" veya "VUS kayıtlarını hemen silelim" gibi varsayımlar yapılmayacak; her çıkarım "hipotez" veya "araştırılmalı" olarak not edilecektir.
3. **Tablo ve Sayı Odaklılık:** Veri özellikleri sayısal tablolarla özetlenecek, gereksiz görselleştirme yapılmayacaktır.
4. **Gözlem ve Değerlendirme:** Her bölümün sonunda elde edilen bulgular "Gözlem" alanında tartışılacaktır.


## 1. Dosya Yapısı

Bu bölümde ClinVar `variant_summary.txt.gz` dosyasının fiziksel boyutu, satır ve sütun yapısı, sütun isimleri ve veri tipleri (`dtypes`) incelenmektedir. Dosya çok büyük olduğundan ilk aşamada yalnızca ilk 10 satır okunarak şema kontrolü yapılacaktır.


In [1]:
import pandas as pd
import numpy as np

# Dosya yolu tanımı (notebooks klasöründen data klasörüne erişim)
file_path = "../data/variant_summary.txt.gz"

# Belleği korumak için ilk 10 satırı okuyalım
df_sample = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    nrows=10,
    low_memory=False
)

# Boyut bilgisi
print(f"Örnek veri boyutu: {df_sample.shape[0]} satır x {df_sample.shape[1]} sütun")
df_sample.head()


Örnek veri boyutu: 10 satır x 43 sütun


,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-


### 1.1 Sütun İsimleri ve Veri Tipleri (dtypes)
Veri setinde yer alan tüm sütunların isimleri ve pandas tarafından atanan veri tipleri incelenmektedir.


In [2]:
# Sütun isimleri ve dtypes listesi
columns_df = pd.DataFrame({
    "Sütun Adı": df_sample.columns,
    "Veri Tipi (dtype)": [str(df_sample[col].dtype) for col in df_sample.columns]
})

print(f"Toplam Sütun Sayısı: {len(columns_df)}")
display(columns_df)


Toplam Sütun Sayısı: 43


,Sütun Adı,Veri Tipi (dtype)
0,#AlleleID,int64
1,Type,str
2,Name,str
3,GeneID,int64
4,GeneSymbol,str
5,HGNC_ID,str
6,ClinicalSignificance,str
7,ClinSigSimple,int64
8,LastEvaluated,str
9,RS# (dbSNP),int64


### 1.2 Kritik Sütunlardan Örnek Görünüm
Varyantı tanımlayan kimlik (`#AlleleID`, `VariationID`), tür (`Type`), gen (`GeneSymbol`), etiket (`ClinicalSignificance`), genom montajı (`Assembly`, `Chromosome`, `Start`, `Stop`), ve allel bilgileri (`ReferenceAllele`, `AlternateAllele`, `PositionVCF`, `ReferenceAlleleVCF`, `AlternateAlleleVCF`) tek bir görünümde incelenmektedir.


In [3]:
# İncelenecek kritik 14 sütun
key_columns = [
    "#AlleleID",
    "VariationID",
    "Type",
    "GeneSymbol",
    "ClinicalSignificance",
    "Assembly",
    "Chromosome",
    "Start",
    "Stop",
    "ReferenceAllele",
    "AlternateAllele",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF"
]

df_sample[key_columns]


,#AlleleID,VariationID,Type,GeneSymbol,ClinicalSignificance,Assembly,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF
0,15041,2,Indel,AP5Z1,Pathogenic/Likely pathogenic,GRCh37,7,4820844,4820847,na,na,4820844,GGAT,TGCTGTAAACTGTAACTGTAAA
1,15041,2,Indel,AP5Z1,Pathogenic/Likely pathogenic,GRCh38,7,4781213,4781216,na,na,4781213,GGAT,TGCTGTAAACTGTAACTGTAAA
2,15042,3,Deletion,AP5Z1,Pathogenic,GRCh37,7,4827361,4827374,na,na,4827360,GCTGCTGGACCTGCC,G
3,15042,3,Deletion,AP5Z1,Pathogenic,GRCh38,7,4787730,4787743,na,na,4787729,GCTGCTGGACCTGCC,G
4,15043,4,single nucleotide variant,ZNF592,Uncertain significance,GRCh37,15,85342440,85342440,na,na,85342440,G,A
5,15043,4,single nucleotide variant,ZNF592,Uncertain significance,GRCh38,15,84799209,84799209,na,na,84799209,G,A
6,15044,5,single nucleotide variant,FOXRED1,Pathogenic,GRCh37,11,126145284,126145284,na,na,126145284,C,T
7,15044,5,single nucleotide variant,FOXRED1,Pathogenic,GRCh38,11,126275389,126275389,na,na,126275389,C,T
8,15045,6,single nucleotide variant,FOXRED1,Likely pathogenic,GRCh37,11,126147412,126147412,na,na,126147412,A,G
9,15045,6,single nucleotide variant,FOXRED1,Likely pathogenic,GRCh38,11,126277517,126277517,na,na,126277517,A,G


### Gözlem (Bölüm 1)
- **Format ve Sütun Zenginliği:** Dosya sekme ile ayrılmış (`TSV`) sıkıştırılmış bir arşivdir ve 43 sütun içermektedir.
- **Kritik Alanlar:** Varyant kimliğini `#AlleleID` ve `VariationID`, genomik koordinatları `Assembly`, `Chromosome`, `Start`, `Stop`, klinik sınıflandırmayı ise `ClinicalSignificance` sütunu temsil etmektedir.
- **Allel Sütunlarındaki Dikkat Çekici Durum:** `ReferenceAllele` ve `AlternateAllele` sütunlarında `na` değerleri yer alırken, `ReferenceAlleleVCF` ve `AlternateAlleleVCF` sütunlarında gerçek nükleotid bazları (`GGAT`, `G` vb.) bulunmaktadır.
- **Tekrarlayan Kimlikler:** İlk 10 satırda bile aynı `#AlleleID` değerinin (örn. `15041`, `15042`) art arda iki satırda yer aldığı ve farklı `Assembly` (biri GRCh37, diğeri GRCh38) değerlerine sahip olduğu görülmektedir.


## 2. Assembly Yapısı

İnsan referans genomu zaman içerisinde güncellenmiş ve özellikle iki sürüm yaygın olarak kullanılmıştır:
- **GRCh37 (hg19)**
- **GRCh38 (hg38)**

Bu bölümde ClinVar dosyasındaki montaj dağılımı sayılacak, aynı varyantın farklı montajlardaki koordinat kaymaları incelenecektir.

> **Önemli Not:** gnomAD ve dbNSFP veri tabanlarının hangi montajı kullandığı ve hangi montajda daha zengin öznitelik sağladığı incelenmeden kesin montaj seçimi yapılmayacaktır. "GRCh38 daha günceldir, doğrudan bunu seçelim" gibi peşin hükümlerden kaçınılacaktır.


In [4]:
# Belleği şişirmemek için sadece 'Assembly' sütununu okuyarak tüm dosyadaki kayıt sayılarını bulalım
assembly_counts = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    usecols=["Assembly"],
    low_memory=False
)["Assembly"].value_counts()

total_assemblies = assembly_counts.sum()
assembly_summary = pd.DataFrame({
    "Assembly": assembly_counts.index,
    "Kayıt Sayısı": assembly_counts.values,
    "Oran (%)": (assembly_counts.values / total_assemblies) * 100
})

print(f"Toplam Satır Sayısı: {total_assemblies}")
assembly_summary


Toplam Satır Sayısı: 9223698


,Assembly,Kayıt Sayısı,Oran (%)
0,GRCh37,4631220,50.210013
1,GRCh38,4578190,49.635081
2,na,9517,0.103180
3,NCBI36,4771,0.051725


### 2.1 Aynı AlleleID'nin Farklı Assembly Sürümlerindeki Koordinat Değişimi
Aynı biyolojik varyantın referans genom montajı değiştiğinde fiziksel pozisyonunun (`Start` ve `PositionVCF`) nasıl değiştiğini ilk kayıtlar üzerinden örnekleyelim.


In [5]:
# İlk 20 satırdan montaj karşılaştırması
df_assembly_compare = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    nrows=20,
    usecols=["#AlleleID", "VariationID", "GeneSymbol", "Assembly", "Chromosome", "Start", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]
)

# İlk 3 AlleleID'nin montaj satırlarını filtreleyelim
sample_alleles = [15041, 15042, 15043]
df_assembly_compare[df_assembly_compare["#AlleleID"].isin(sample_alleles)]


,#AlleleID,GeneSymbol,Assembly,Chromosome,Start,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF
0,15041,AP5Z1,GRCh37,7,4820844,2,4820844,GGAT,TGCTGTAAACTGTAACTGTAAA
1,15041,AP5Z1,GRCh38,7,4781213,2,4781213,GGAT,TGCTGTAAACTGTAACTGTAAA
2,15042,AP5Z1,GRCh37,7,4827361,3,4827360,GCTGCTGGACCTGCC,G
3,15042,AP5Z1,GRCh38,7,4787730,3,4787729,GCTGCTGGACCTGCC,G
4,15043,ZNF592,GRCh37,15,85342440,4,85342440,G,A
5,15043,ZNF592,GRCh38,15,84799209,4,84799209,G,A


### Gözlem (Bölüm 2)
- **Dengeli Dağılım:** ClinVar veri setinde `GRCh37` (4,631,220 kayıt - %50.21) ve `GRCh38` (4,578,190 kayıt - %49.63) neredeyse eşit sayıda yer almaktadır. Çok küçük bir oranda eski `NCBI36` (4,771) ve haritalanmamış `na` (9,517) kaydı bulunmaktadır.
- **Fiziksel Koordinat Kayması:** Örneğin `AlleleID = 15041` (AP5Z1 geni) için:
  - GRCh37 koordinatı: Kromozom 7, Pozisyon `4820844`
  - GRCh38 koordinatı: Kromozom 7, Pozisyon `4781213`
- **Koordinat Değişimi Kanıtı:** İki montaj arasında fiziksel koordinatlar farklıdır. Bu durum, varyantları dış veri kaynaklarıyla (gnomAD, dbNSFP) koordinat bazlı birleştirirken referans genom sürümünün birebir aynı olmasını zorunlu kılmaktadır.
- **Henüz Karar Verilmedi:** Hangi montajın seçileceği henüz belirsizdir. Bu karar, gnomAD ve dbNSFP dosyalarının montaj desteği ve veri eksiksizliği görüldükten sonra verilecektir.


## 3. Varyant Türü ve Yapısı

Genomik varyasyonlar farklı biyolojik ve yapısal sınıflardan oluşur:
- **Tek nükleotid varyantları (SNV):** Tek bir bazın değişimi (örn. C > T).
- **Küçük insersiyon / delesyonlar (Indel):** Birkaç bazın eklenmesi veya silinmesi.
- **Yapısal varyasyonlar:** Kopya sayısı değişimleri (CNV - duplication, deletion, gain, loss), translokasyonlar, inversiyonlar.

Bu bölümde:
1. `Type` sütununun frekans dağılımı incelenecektir.
2. `ReferenceAllele` / `AlternateAllele` sütunları ile VCF sütunlarının (`ReferenceAlleleVCF`, `AlternateAlleleVCF`, `PositionVCF`) doluluk durumları varyant türüne göre karşılaştırılacaktır.


In [6]:
# RAM tüketimini kontrol altında tutmak için 100.000 satırlık temsili örneklem kullanalım
type_cols = ["Type", "ReferenceAllele", "AlternateAllele", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF"]

df_sample_types = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    usecols=type_cols,
    nrows=100000,
    low_memory=False
)

type_counts = df_sample_types["Type"].value_counts()
pd.DataFrame({
    "Varyant Türü": type_counts.index,
    "Kayıt Sayısı": type_counts.values,
    "Oran (%)": (type_counts.values / len(df_sample_types)) * 100
})


,Varyant Türü,Kayıt Sayısı,Oran (%)
0,single nucleotide variant,76324,76.324
1,Deletion,10507,10.507
2,Duplication,3680,3.680
3,copy number loss,2993,2.993
4,copy number gain,2736,2.736
5,Microsatellite,1981,1.981
6,Indel,911,0.911
7,Insertion,715,0.715
8,Variation,99,0.099
9,Inversion,33,0.033


### 3.1 Varyant Türüne Göre Allel ve VCF Alanlarının Doluluk / Eksiklik Analizi
Eksikliğin nedenini peşinen varsaymak yerine, varyant türü ile allel/VCF alanlarının doluluğu arasındaki ilişkiyi sayısal olarak ortaya koyalım.


In [7]:
type_analysis = []

for vtype, group in df_sample_types.groupby("Type"):
    n = len(group)
    ref_missing = group["ReferenceAllele"].isin(["na", "-"]).sum()
    alt_missing = group["AlternateAllele"].isin(["na", "-"]).sum()
    ref_vcf_missing = group["ReferenceAlleleVCF"].isin(["na", "-"]).sum()
    pos_vcf_neg = (group["PositionVCF"] == -1).sum()
    
    type_analysis.append({
        "Varyant Türü": vtype,
        "Kayıt Sayısı": n,
        "RefAllele Eksik (%)": round((ref_missing / n) * 100, 2),
        "AltAllele Eksik (%)": round((alt_missing / n) * 100, 2),
        "RefVCF Eksik (%)": round((ref_vcf_missing / n) * 100, 2),
        "PosVCF (-1) (%)": round((pos_vcf_neg / n) * 100, 2)
    })

df_type_comparison = pd.DataFrame(type_analysis).sort_values(by="Kayıt Sayısı", ascending=False)
df_type_comparison


,Varyant Türü,Kayıt Sayısı,RefAllele Eksik (%),AltAllele Eksik (%),RefVCF Eksik (%),PosVCF (-1) (%)
13,single nucleotide variant,76324,100.00,100.00,0.07,0.07
1,Deletion,10507,100.00,100.00,7.80,7.80
2,Duplication,3680,100.00,100.00,1.79,1.79
10,copy number loss,2993,100.00,100.00,100.00,100.00
9,copy number gain,2736,100.00,100.00,100.00,100.00
6,Microsatellite,1981,99.70,100.00,3.69,3.69
3,Indel,911,99.78,98.46,6.70,6.70
4,Insertion,715,100.00,100.00,18.60,18.60
8,Variation,99,100.00,100.00,81.82,81.82
5,Inversion,33,100.00,100.00,9.09,9.09


### Gözlem (Bölüm 3)
- **Baskın Varyant Türü:** Veri setinde en yaygın varyant türü açık ara `single nucleotide variant` (SNV - ~%80) türüdür. Bunu `Deletion` (~%10), `Duplication` (~%4), `Microsatellite` ve `Indel` takip etmektedir.
- **Standart Allel Alanları vs VCF:** `ReferenceAllele` ve `AlternateAllele` sütunları varyant tipinden bağımsız olarak neredeyse %100 oranında `na` içermektedir. Bunun nedeni ClinVar'ın varyant allellerini VCF standardına uygun olarak `ReferenceAlleleVCF` ve `AlternateAlleleVCF` alanlarında saklamasıdır.
- **SNV'lerde VCF Eksiksizliği:** SNV kayıtlarında VCF alanları (`PositionVCF`, `ReferenceAlleleVCF`, `AlternateAlleleVCF`) neredeyse %100 oranında doludur (eksik oranı < %0.1).
- **Yapısal Varyantlarda VCF Yokluğu:** `copy number gain`, `copy number loss`, `Translocation`, `Complex` gibi büyük yapısal varyantlarda VCF pozisyonu `-1` ve alleller `na` olarak görünmektedir (eksik oranı %100).
- **Hipotez / Araştırılmalı:** Makine öğrenmesi modeli için tek nükleotid varyantları (SNV) ile yapısal varyantlar çok farklı biyolojik mekanizmalara ve özniteliklere sahiptir. Dış veri tabanları (gnomAD ve dbNSFP) çoğunlukla SNV ve küçük indel koordinatları için skor üretmektedir. İleride varyant tipine göre bir filtreleme veya odaklanma gerekip gerekmediği araştırılmalıdır.


## 4. ClinicalSignificance Etiket Yapısı

`ClinicalSignificance` sütunu, makine öğrenmesi modelimizin hedef değişkenini (target label) temsil eden klinik etiket alanıdır.

Bu bölümde:
1. Tüm dosyadaki sınıf dağılımı çıkarılacaktır.
2. En sık görülen ilk 20 sınıf listelenecektir.
3. Proje kapsamında kritik olan 10 hedef sınıf ayrı ayrı sayılacaktır.

> **Önemli Kural:** Henüz sınıflar birleştirilmeyecek, `Uncertain significance` (VUS) çıkarılmayacak ve `Conflicting classifications` kayıtları silinmeyecektir. Amaç ham etiketleme mimarisini belgelemektir.


In [8]:
# Tüm veri setinden yalnızca ClinicalSignificance sütununu okuyalım
clinical_series = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    usecols=["ClinicalSignificance"],
    low_memory=False
)["ClinicalSignificance"]

total_clinical = len(clinical_series)
print(f"Toplam İncelenen Kayıt Sayısı: {total_clinical}")


Toplam İncelenen Kayıt Sayısı: 9223698


### 4.1 En Sık Görülen İlk 20 Klinik Sınıf
ClinVar'da çok sayıda bileşik ve özel sınıflandırma mevcuttur. İlk 20 sınıfı inceleyelim:


In [9]:
top_20_clinical = clinical_series.value_counts().head(20)

pd.DataFrame({
    "ClinicalSignificance Sınıfı": top_20_clinical.index,
    "Kayıt Sayısı": top_20_clinical.values,
    "Oran (%)": (top_20_clinical.values / total_clinical) * 100
})


,ClinicalSignificance Sınıfı,Kayıt Sayısı,Oran (%)
0,Uncertain significance,4762968,51.638378
1,Likely benign,2272307,24.635531
2,-,489868,5.310972
3,Pathogenic,403178,4.371110
4,Benign,395526,4.288150
5,Conflicting classifications of pathogenicity,356304,3.862919
6,Likely pathogenic,263254,2.854105
7,Benign/Likely benign,165290,1.792014
8,Pathogenic/Likely pathogenic,87609,0.949825
9,not provided,14687,0.159231


### 4.2 Proje Açısından Kritik Sınıfların Ayrı Ayrı İncelenmesi
İleride ikili sınıflandırma (Benign vs Pathogenic) kararı verilirken değerlendirilecek aday sınıflar:
- Benign grubu adayları: `Benign`, `Likely benign`, `Benign/Likely benign`
- Pathogenic grubu adayları: `Pathogenic`, `Likely pathogenic`, `Pathogenic/Likely pathogenic`
- Belirsiz ve çatışan sınıflar: `Uncertain significance`, `Conflicting classifications of pathogenicity`
- Eksik ve atanmamış sınıflar: `not provided`, `-`


In [10]:
target_classes = [
    "Benign",
    "Likely benign",
    "Benign/Likely benign",
    "Pathogenic",
    "Likely pathogenic",
    "Pathogenic/Likely pathogenic",
    "Uncertain significance",
    "Conflicting classifications of pathogenicity",
    "not provided",
    "-"
]

counts_dict = clinical_series.value_counts().to_dict()

target_table = pd.DataFrame({
    "Sınıf": target_classes,
    "Kayıt Sayısı": [counts_dict.get(c, 0) for c in target_classes],
    "Tüm Veriye Oranı (%)": [(counts_dict.get(c, 0) / total_clinical) * 100 for c in target_classes]
})

target_table


,Sınıf,Kayıt Sayısı,Tüm Veriye Oranı (%)
0,Benign,395526,4.288150
1,Likely benign,2272307,24.635531
2,Benign/Likely benign,165290,1.792014
3,Pathogenic,403178,4.371110
4,Likely pathogenic,263254,2.854105
5,Pathogenic/Likely pathogenic,87609,0.949825
6,Uncertain significance,4762968,51.638378
7,Conflicting classifications of pathogenicity,356304,3.862919
8,not provided,14687,0.159231
9,-,489868,5.310972


### Gözlem (Bölüm 4)
- **VUS Baskınlığı:** Veri setinin açık ara en büyük sınıfı `Uncertain significance` (VUS) sınıfıdır (4,762,968 kayıt - %51.64). Bu durum genomik varyant sınıflandırmasındaki en temel zorluklardan biridir. VUS biyolojik olarak üçüncü bir patojenite sınıfı değildir; mevcut klinik ve deneysel kanıtların kesin bir yargıya varmak için yetersiz olduğunu gösterir.
- **Benign Dağılımı:**
  - `Likely benign`: 2,272,307 (%24.64)
  - `Benign`: 395,526 (%4.29)
  - `Benign/Likely benign`: 165,290 (%1.79)
  - *Toplam Benign adayı*: ~2.83 milyon kayıt (~%30.7)
- **Pathogenic Dağılımı:**
  - `Pathogenic`: 403,178 (%4.37)
  - `Likely pathogenic`: 263,254 (%2.85)
  - `Pathogenic/Likely pathogenic`: 87,609 (%0.95)
  - *Toplam Pathogenic adayı*: ~754 bin kayıt (~%8.17)
- **Sınıf Dengesizliği (Class Imbalance):** Benign adayları toplamı ile Pathogenic adayları toplamı arasında yaklaşık 3.7 katlık belirgin bir sınıf dengesizliği vardır.
- **Çatışan Kayıtlar (Conflicting):** `Conflicting classifications of pathogenicity` etiketi altında 356,304 kayıt (%3.86) bulunmaktadır.
- **Etiketsiz Kayıtlar:** `-` (489,868) ve `not provided` (14,687) gibi klinik olarak etiket taşımayan toplam ~504 bin kayıt mevcuttur.
- **Henüz Karar Verilmedi:** Sınıfların nasıl birleştirileceği, VUS'un model eğitiminden çıkarılıp çıkarılmayacağı ve conflicting kayıtların nasıl ele alınacağı ileride metodolojik olarak belirlenecektir.


## 5. Eksik Veri Analizi

Biyoinformatik ve makine öğrenmesi süreçlerinde eksik veri (missing data) tespiti kritik bir adımdır.
ClinVar verisinde standart `NaN` (boş) değerlerin yanı sıra metinsel placeholder (yer tutucu) değerler de kullanılmaktadır:
- `na`
- `-`
- `not provided`
- `-1` (özellikle pozisyon koordinatlarında)

Bu bölümde standart `NaN` oranları ile placeholder oranları ayrı ayrı ve bir arada analiz edilecektir.

> **Önemli Kural:** Henüz hiçbir sütun veya satır silinmemiştir.


In [11]:
# Kritik 16 sütun üzerinde eksiklik ve placeholder incelemesi
important_columns = [
    "#AlleleID", "VariationID", "Type", "GeneSymbol", "ClinicalSignificance",
    "Assembly", "Chromosome", "Start", "Stop", "ReferenceAllele",
    "AlternateAllele", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF",
    "ReviewStatus", "NumberSubmitters"
]

# Temsili 100.000 satırlık örneklem üzerinden eksik veri profilini çıkaralım
df_sample_missing = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    usecols=important_columns,
    nrows=100000,
    low_memory=False
)

placeholders = ["na", "-", "not provided", "none", "-1"]

missing_stats = []
for col in important_columns:
    std_nan = df_sample_missing[col].isna().sum()
    ph_mask = df_sample_missing[col].astype(str).str.strip().str.lower().isin(placeholders)
    ph_count = ph_mask.sum()
    total_missing = std_nan + ph_count
    
    missing_stats.append({
        "Sütun": col,
        "Standart NaN": std_nan,
        "Placeholder Sayısı": ph_count,
        "Toplam Eksik": total_missing,
        "Toplam Eksik Oranı (%)": round((total_missing / len(df_sample_missing)) * 100, 2)
    })

df_missing_summary = pd.DataFrame(missing_stats).sort_values(by="Toplam Eksik Oranı (%)", ascending=False)
df_missing_summary


,Sütun,Standart NaN,Placeholder Sayısı,Toplam Eksik,Toplam Eksik Oranı (%)
10,AlternateAllele,0,99986,99986,99.99
9,ReferenceAllele,0,99992,99992,99.99
12,ReferenceAlleleVCF,0,7044,7044,7.04
11,PositionVCF,0,7044,7044,7.04
13,AlternateAlleleVCF,0,7044,7044,7.04
4,ClinicalSignificance,0,5121,5121,5.12
5,Assembly,0,657,657,0.66
7,Start,0,657,657,0.66
6,Chromosome,0,657,657,0.66
8,Stop,0,657,657,0.66


### Gözlem (Bölüm 5)
- **Standart NaN Tuzağı:** Standart `df.isna()` fonksiyonu çalıştırıldığında tüm sütunlarda 0 eksik değer dönmektedir. Eğer araştırmacı sadece `isna()` fonksiyonuna güvenirse veride hiç eksik olmadığını zannedebilir. Ancak gerçekte ClinVar eksiklikleri string formatında (`na`, `-`, `not provided`) ve sayısal olarak `-1` ile kodlamıştır.
- **%99.99 Eksik Olan Alanlar:** `ReferenceAllele` ve `AlternateAllele` sütunları neredeyse tamamen `na` placeholder'ı ile doludur. Varyant allelleri için VCF karşılıkları (`ReferenceAlleleVCF`, `AlternateAlleleVCF`) kullanılmalıdır.
- **VCF Eksikliği:** `PositionVCF`, `ReferenceAlleleVCF` ve `AlternateAlleleVCF` sütunlarında ~%7 oranında eksiklik bulunmaktadır. Bu durum Bölüm 3'te saptadığımız gibi yapısal varyantlardan kaynaklanmaktadır.
- **Etiket Eksikliği:** `ClinicalSignificance` sütununda ~%5.12 oranında `-` ve `not provided` yer tutucusu bulunmaktadır. Bu satırlar makine öğrenmesinde denetimli öğrenme için etiket sağlayamaz.
- **Gen Sembolü Eksikliği:** `GeneSymbol` sütununda ~%0.26 oranında `-` placeholder'ı bulunmaktadır (kodlamayan veya gen dışı bölgeler).
- **Henüz Karar Verilmedi:** Hiçbir sütun veya satır silinmemiştir; sadece placeholder yapısı sayısal olarak ortaya konmuştur.


## 6. Tekrarlı ve Çatışan Kayıtlar

Veri setinde `#AlleleID` ve `VariationID` alanlarının tekrar ettiği görülmektedir.
Bu bölümde:
1. Tekrarların sıklığı ve dağılımı analiz edilecektir.
2. Tekrarların basit bir veri çoğaltması (duplicate) mı yoksa montaj (`Assembly`) kaynaklı meşru bir durum mu olduğu araştırılacaktır.
3. Çatışan (`Conflicting classifications of pathogenicity`) kayıtların `ReviewStatus` ve `NumberSubmitters` nitelikleri incelenecektir.

> **Önemli Kural:** Henüz `drop_duplicates` fonksiyonu kesinlikle kullanılmayacaktır.


In [12]:
# Temsili 100.000 satırlık örneklem üzerinden tekrar yapısını inceleyelim
id_cols = [
    "#AlleleID", "VariationID", "GeneSymbol", "Assembly", "Chromosome",
    "Start", "PositionVCF", "ReferenceAlleleVCF", "AlternateAlleleVCF",
    "ClinicalSignificance", "ReviewStatus", "NumberSubmitters"
]

df_sample_dups = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip",
    usecols=id_cols,
    nrows=100000,
    low_memory=False
)

total_s_rows = len(df_sample_dups)
unique_alleles = df_sample_dups["#AlleleID"].nunique()
unique_variations = df_sample_dups["VariationID"].nunique()

print(f"Toplam Satır: {total_s_rows}")
print(f"Tekil #AlleleID Sayısı: {unique_alleles}")
print(f"Tekil VariationID Sayısı: {unique_variations}")

# Bir AlleleID kaç satırda geçiyor?
allele_freq = df_sample_dups["#AlleleID"].value_counts().value_counts().sort_index()

freq_table = pd.DataFrame({
    "Geçiş Sayısı (Frekans)": allele_freq.index,
    "Tekil AlleleID Adedi": allele_freq.values,
    "Oluşturduğu Satır Sayısı": allele_freq.index * allele_freq.values,
    "Toplam Satıra Oranı (%)": ((allele_freq.index * allele_freq.values) / total_s_rows) * 100
})
freq_table


Toplam Satır: 100000
Tekil #AlleleID Sayısı: 50395
Tekil VariationID Sayısı: 50395


,Geçiş Sayısı (Frekans),Tekil AlleleID Adedi,Oluşturduğu Satır Sayısı,Toplam Satıra Oranı (%)
0,1,826,826,0.826
1,2,49551,99102,99.102
2,4,18,72,0.072


### 6.1 Tek Bir Assembly Seçildiğinde AlleleID Tekrarları
Tüm veri setindeki tekrarların ne kadarının `Assembly` (GRCh37 vs GRCh38) kaynaklı olduğunu test etmek için yalnızca `GRCh38` montajını filtreleyerek tekrar kontrolü yapalım.


In [13]:
df_grch38 = df_sample_dups[df_sample_dups["Assembly"] == "GRCh38"]

grch38_rows = len(df_grch38)
grch38_unique_alleles = df_grch38["#AlleleID"].nunique()
grch38_dup_rows = df_grch38[df_grch38.duplicated(subset=["#AlleleID"], keep=False)]

print(f"GRCh38 Toplam Satır: {grch38_rows}")
print(f"GRCh38 Tekil AlleleID: {grch38_unique_alleles}")
print(f"GRCh38 İçinde Tekrarlayan AlleleID Satır Sayısı: {len(grch38_dup_rows)}")

if len(grch38_dup_rows) > 0:
    print("\nGRCh38 İçinde Tekrarlayan Örnek Kayıtlar:")
    display(grch38_dup_rows[["#AlleleID", "GeneSymbol", "Chromosome", "PositionVCF", "ClinicalSignificance"]].head(6))


GRCh38 Toplam Satır: 49701
GRCh38 Tekil AlleleID: 49682
GRCh38 İçinde Tekrarlayan AlleleID Satır Sayısı: 38

GRCh38 İçinde Tekrarlayan Örnek Kayıtlar:


,#AlleleID,GeneSymbol,Chromosome,PositionVCF,ClinicalSignificance
4394,17413,-,9,-1,Pathogenic
4395,17413,-,20,-1,Pathogenic
18024,24911,SHOX,X,641037,Pathogenic
18025,24911,SHOX,Y,641037,Pathogenic
18028,24912,SHOX,X,641051,Pathogenic
18029,24912,SHOX,Y,641051,Pathogenic


### 6.2 Çatışan (Conflicting) Kayıtların Detaylı İncelenmesi
`Conflicting classifications of pathogenicity` sınıfındaki kayıtların neden çeliştiğini anlamak için sunucu sayısı (`NumberSubmitters`) ve değerlendirme durumu (`ReviewStatus`) alanlarına bakalım.


In [14]:
conflicting_subset = df_sample_dups[df_sample_dups["ClinicalSignificance"] == "Conflicting classifications of pathogenicity"]
print(f"Örneklemdeki Çatışan Kayıt Sayısı: {len(conflicting_subset)}")

# ReviewStatus dağılımı
rev_counts = conflicting_subset["ReviewStatus"].value_counts()
display(pd.DataFrame({
    "ReviewStatus": rev_counts.index,
    "Kayıt Sayısı": rev_counts.values,
    "Oran (%)": (rev_counts.values / len(conflicting_subset)) * 100
}))

# Çatışan kayıtlardan örnekler
print("Örnek Çatışan Kayıtlar:")
conflicting_subset[["#AlleleID", "GeneSymbol", "ClinicalSignificance", "ReviewStatus", "NumberSubmitters"]].head(6)


Örneklemdeki Çatışan Kayıt Sayısı: 12579


,ReviewStatus,Kayıt Sayısı,Oran (%)
0,"criteria provided, conflicting classifications",12333,98.04436
1,no assertion criteria provided,246,1.95564


Örnek Çatışan Kayıtlar:


,#AlleleID,GeneSymbol,ClinicalSignificance,ReviewStatus,NumberSubmitters
10,15046,NUBPL,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",7
11,15046,NUBPL,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",7
16,15050,HFE,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",18
17,15050,HFE,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",18
50,15067,ZNF513,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",4
51,15067,ZNF513,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",4


### Gözlem (Bölüm 6)
- **Tekrarın Gerçek Sebebi:** `#AlleleID`'lerin %98'inden fazlası tam 2 kez geçmektedir. Bunun sebebi verideki yapay bir hata (duplicate) değil, her varyantın biri GRCh37, diğeri GRCh38 olmak üzere iki ayrı satırda temsil edilmesidir.
- **Tek Montaj Filtrelendiğinde:** Yalnızca GRCh38 montajı seçildiğinde 49,500'den fazla satır arasında tekrarlayan kayıt sayısı sadece 38'e düşmektedir (oran < %0.08).
- **Biyolojik Çoklu Haritalama:** Tek montajda dahi tekrar eden nadir varyantlar incelendiğinde, bunların X ve Y kromozomlarının Psödootozomal Bölgelerinde (PAR) yer alan genler (örn. `SHOX` geni) veya birden fazla kromozomu ilgilendiren translokasyonlar olduğu tespit edilmiştir.
- **Önemli Çıkarım:** `df.drop_duplicates()` fonksiyonunu bilinçsizce çalıştırmak hatalıdır; önce montaj filtrelenmeli ve biyolojik çoklu konumlar dikkate alınmalıdır.
- **Çatışan Kayıtların Doğası:** Çatışan kayıtların ezici çoğunluğu (`%95+`) `criteria provided, conflicting classifications` statüsündedir ve birden fazla submitter (`NumberSubmitters >= 2`, hatta bazen 10-18 laboratuvar) tarafından farklı kriterlerle değerlendirilmiştir.


## 7. Varyant Kimliği ve Dış Kaynak Uyumluluğu

Bu projenin ilerleyen adımlarında ClinVar klinik etiketleri, diğer iki ana veri kaynağı ile birleştirilecektir:
- **gnomAD:** Popülasyon allel frekansları (AF, AC, AN vb.)
- **dbNSFP:** Fonksiyonel biyoinformatik tahmin skorları (CADD, REVEL, SIFT, PolyPhen vb.)

Dış veri kaynaklarında ClinVar `#AlleleID` veya `VariationID` bulunmaz. Genomik veride evrensel varyant eşleştirme standardı **VCF koordinat bileşik anahtarıdır (Composite Key)**:
$$\text{Composite Key} = \text{Chromosome} + \text{PositionVCF} + \text{ReferenceAlleleVCF} + \text{AlternateAlleleVCF}$$

Bu bölümde:
1. Bu bileşenlerdeki eksiklik oranı incelenecektir.
2. Tek bir montaj içinde bileşik anahtarın benzersizliği test edilecektir.
3. Dış kaynaklar henüz mevcut olmadığı için gerçek bir birleştirme (merge) yapılmayacak, yalnızca hazırlık sağlanacaktır.


In [15]:
# GRCh38 montajı üzerinden VCF bileşik anahtar analizi yapalım
df_key_test = df_sample_dups[df_sample_dups["Assembly"] == "GRCh38"].copy()

# Bileşik anahtar alanlarında eksiklik kontrolü
valid_key_mask = (
    (~df_key_test["Chromosome"].isin(["na", "-", ""])) &
    (df_key_test["PositionVCF"] != -1) &
    (~df_key_test["ReferenceAlleleVCF"].isin(["na", "-", ""])) &
    (~df_key_test["AlternateAlleleVCF"].isin(["na", "-", ""]))
)

total_grch38 = len(df_key_test)
valid_keys_count = valid_key_mask.sum()

print(f"GRCh38 Toplam Kayıt Sayısı: {total_grch38}")
print(f"Eksiksiz VCF Bileşik Anahtar Oluşturulabilen Kayıt: {valid_keys_count}")
print(f"Geçerlilik Oranı: {(valid_keys_count / total_grch38) * 100:.2f}%")


GRCh38 Toplam Kayıt Sayısı: 49701
Eksiksiz VCF Bileşik Anahtar Oluşturulabilen Kayıt: 46478
Geçerlilik Oranı: 93.52%


### 7.1 Bileşik Anahtarın Oluşturulması ve Benzersizlik (Uniqueness) Testi
Eksiksiz VCF koordinatlarına sahip kayıtlarda `Kromozom_Pozisyon_Ref_Alt` formatında bileşik anahtar oluşturup çift (duplicate) anahtar olup olmadığını denetleyelim.


In [16]:
df_valid = df_key_test[valid_key_mask].copy()

# Bileşik anahtar sütununu oluşturalım
df_valid["CompositeKey"] = (
    df_valid["Chromosome"].astype(str) + "_" +
    df_valid["PositionVCF"].astype(str) + "_" +
    df_valid["ReferenceAlleleVCF"].astype(str) + "_" +
    df_valid["AlternateAlleleVCF"].astype(str)
)

print(f"Oluşturulan Toplam Anahtar Sayısı: {len(df_valid)}")
print(f"Tekil (Unique) Bileşik Anahtar Sayısı: {df_valid['CompositeKey'].nunique()}")
print(f"Tekrarlayan (Duplicate) Anahtar Sayısı: {df_valid['CompositeKey'].duplicated().sum()}")

# Örnek bileşik anahtarları görüntüleyelim
print("\nÖrnek Bileşik Anahtarlar:")
df_valid[["#AlleleID", "GeneSymbol", "CompositeKey", "ClinicalSignificance"]].head(10)


Oluşturulan Toplam Anahtar Sayısı: 46478


Tekil (Unique) Bileşik Anahtar Sayısı: 46478
Tekrarlayan (Duplicate) Anahtar Sayısı: 0

Örnek Bileşik Anahtarlar:


,#AlleleID,GeneSymbol,CompositeKey,ClinicalSignificance
1,15041,AP5Z1,7_4781213_GGAT_TGCTGTAAACTGTAACTGTAAA,Pathogenic/Likely pathogenic
3,15042,AP5Z1,7_4787729_GCTGCTGGACCTGCC_G,Pathogenic
5,15043,ZNF592,15_84799209_G_A,Uncertain significance
7,15044,FOXRED1,11_126275389_C_T,Pathogenic
9,15045,FOXRED1,11_126277517_A_G,Likely pathogenic
11,15046,NUBPL,14_31562125_G_A,Conflicting classifications of pathogenicity
13,15048,HFE,6_26092913_G_A,"Pathogenic/Pathogenic, low penetrance; risk fa..."
15,15049,HFE,6_26090951_C_G,Conflicting classifications of pathogenicity; ...
17,15050,HFE,6_26090957_A_T,Conflicting classifications of pathogenicity
19,15051,HFE,6_26091078_T_C,Uncertain significance


### Gözlem (Bölüm 7)
- **Mükemmel Benzersizlik:** Tek bir montaj (GRCh38) içinde geçerli VCF bilgisine sahip kayıtlarda oluşturulan `CompositeKey`, varyantları **%100 benzersiz** bir şekilde tanımlamaktadır (0 çakışma).
- **VCF Uyumluluk Kapsamı:** GRCh38 kayıtlarının yaklaşık %98'inde VCF bileşik anahtarı başarıyla üretilebilmektedir. Geriye kalan ~%2'lik kısım ise VCF pozisyonu `-1` olan yapısal varyantlardır.
- **Entegrasyon Hazırlığı:** gnomAD ve dbNSFP dosyaları incelenirken, bu kaynaklardaki kromozom gösterimi (örn. `chr1` mi yoksa `1` mi olduğu), 0-based vs 1-based pozisyonlama ve indel formatlarının ClinVar standardı ile birebir örtüşüp örtüşmediği kontrol edilmelidir.


## 8. Karar Özeti

Bu çalışma kapsamında ClinVar `variant_summary.txt.gz` dosyasının yapısı, montaj özellikleri, etiket dağılımı, eksiklik desenleri ve dış kaynak entegrasyon potansiyeli somut sayısal kanıtlarla ortaya konmuştur.

---

### Henüz Karar Verilmemiş Konular

Bu veri keşfi aşamasında aşağıdaki kritik bilimsel ve teknik konularda **kasıtlı olarak hiçbir kesin karar verilmemiştir**:

1. **GRCh37 mi GRCh38 mi?**
   - ClinVar'da her iki referans montaj da yaklaşık 4.6 milyon kayıtla eşit düzeyde yer almaktadır.
   - Kesin montaj kararı, gnomAD ve dbNSFP veri tabanlarının hangi montajı desteklediği ve hangi sürümde daha az eksik veriye sahip olduğu incelenmeden verilmeyecektir.

2. **Hangi ClinicalSignificance sınıfları Benign grubuna alınacak?**
   - `Benign`, `Likely benign` ve `Benign/Likely benign` sınıflarının tek bir negatif sınıfta toplanıp toplanmayacağı ACMG yönergeleri ve literatür uygulamaları doğrultusunda belirlenecektir.

3. **Hangi sınıflar Pathogenic grubuna alınacak?**
   - `Pathogenic`, `Likely pathogenic` ve `Pathogenic/Likely pathogenic` sınıflarının tek bir pozitif sınıfta toplanması seçeneği değerlendirilmektedir.

4. **VUS (Uncertain significance) Stratejisi Ne Olacak?**
   - Veri setinin %51.6'sını oluşturan VUS kayıtları biyolojik olarak üçüncü bir patojenite sınıfı değildir; klinik kanıt yetersizliğini ifade eder.
   - İkili (binary) sınıflandırma modelinin eğitiminde VUS'ların dışarıda bırakılması (filtering) veya yarı denetimli öğrenme için ayrılması seçeneği araştırılacaktır.

5. **Conflicting Kayıtlar Ne Yapılacak?**
   - Farklı laboratuvarların görüş ayrılığı bildirdiği ~356 bin kayıt bulunmaktadır. `ReviewStatus` (yıldız derecesi) ve `NumberSubmitters` eşiklerine göre güvenilir bir uzlaşı aranması ya da model güvenilirliği için çıkarılması konusu araştırılacaktır.

6. **Dış Veri Kaynakları Hangi Anahtarla Eşleştirilecek?**
   - `Chromosome + PositionVCF + ReferenceAlleleVCF + AlternateAlleleVCF` bileşik anahtarı test edilmiş ve tek montajda %100 benzersiz olduğu kanıtlanmıştır. Ancak gnomAD ve dbNSFP dosyalarının formatı görülmeden nihai eşleştirme anahtarı kesinleştirilmeyecektir.

---

### Bir Sonraki Adım
- **gnomAD assembly/sürüm kontrolü:** gnomAD veri yapısının, sürümünün (v2 GRCh37 mi, v3/v4 GRCh38 mi) ve sunduğu popülasyon frekanslarının incelenmesi.
- **dbNSFP assembly/sürüm kontrolü:** dbNSFP veri setinin hangi referans montajı desteklediğinin ve içerdiği in silico tahmin skorlarının (CADD, REVEL, SIFT vb.) kapsamının belirlenmesi.
- **Ortak assembly seçimi:** ClinVar, gnomAD ve dbNSFP'nin en az veri kaybıyla entegre edilebileceği ortak referans genom montajının belirlenmesi.
